This is Rachel's version of the notebook. Me and Jah will merge later.

Clean the data by eliminating rides lasting more than three hours and rides with negative duration, if
any. Long rides may correspond to bikes left undocked, and negative durations may reflect recording
errors. Report the number of rows and columns before cleaning, the number of rides removed by
each rule, and the number retained. Document any additional cleaning decisions.
1. Using the start and end times, compute each ride’s duration in minutes and plot a histogram
of ride durations.
2. What is the expected ride duration (the average ride duration)? What is the empirical variance
of ride duration? What is the probability that a ride lasts more than 20 minutes?
3. What is the probability that a ride lasts more than 20 minutes conditional on the user being
a Citi Bike member?
4. Suppose that a ride lasts more than 25 minutes. What is the probability that this ride belongs
to a Citi Bike member?
5. What is the expected ride duration of an electric bike? What is the expected ride duration of
a classic bike?
6. Suppose that a ride lasts less than 10 minutes. What is the probability that it uses an electric
bike? What is the probability that it uses a classic bike? Comment on the results.

In [1]:
from pathlib import Path
import pandas as pd

input_dir = Path("202608-citibike-tripdata")
output_dir = Path("clean-202608-citibike-tripdata")
source_files = sorted(input_dir.glob("*.csv"))

if not source_files:
    raise FileNotFoundError(f"No CSV files found in {input_dir}")

output_dir.mkdir(parents=True, exist_ok=True)
chunk_size = 250_000
totals = {"rows_before": 0, "over_3_hours": 0, "negative_duration": 0, "rows_retained": 0}

for source_file in source_files:
    output_file = output_dir / f"{source_file.stem}_cleaned.csv"
    file_totals = {"rows_before": 0, "over_3_hours": 0, "negative_duration": 0, "rows_retained": 0}

    # Process in chunks so the full set of large CSV files does not need to fit in memory.
    for chunk_number, chunk in enumerate(
        pd.read_csv(source_file, chunksize=chunk_size, low_memory=False)
    ):
        # Calculate ride duration from the recorded start and end timestamps.
        duration = pd.to_datetime(chunk["ended_at"]) - pd.to_datetime(chunk["started_at"])
        over_3_hours = duration > pd.Timedelta(hours=3)
        negative_duration = duration < pd.Timedelta(0)

        # Keep rides from zero through three hours; remove long and negative-duration rides.
        cleaned_chunk = chunk.loc[~(over_3_hours | negative_duration)]

        file_totals["rows_before"] += len(chunk)
        file_totals["over_3_hours"] += int(over_3_hours.sum())
        file_totals["negative_duration"] += int(negative_duration.sum())
        file_totals["rows_retained"] += len(cleaned_chunk)
        cleaned_chunk.to_csv(
            output_file,
            mode="w" if chunk_number == 0 else "a",
            header=chunk_number == 0,
            index=False,
        )

    for key in totals:
        totals[key] += file_totals[key]

    print(
        f"{source_file.name}: {file_totals['rows_before']:,} rows before; "
        f"{file_totals['over_3_hours']:,} over 3 hours removed; "
        f"{file_totals['negative_duration']:,} negative-duration rides removed; "
        f"{file_totals['rows_retained']:,} retained -> {output_file}"
    )

print("\nOverall cleaning summary")
print(f"Rows before cleaning: {totals['rows_before']:,}")
print(f"Columns per input file: {len(pd.read_csv(source_files[0], nrows=0).columns)}")
print(f"Were any original rides over 3 hours? {'Yes' if totals['over_3_hours'] else 'No'}")
print(f"Rides over 3 hours removed: {totals['over_3_hours']:,}")
print(f"Were any original rides negative-duration? {'Yes' if totals['negative_duration'] else 'No'}")
print(f"Rides with negative duration removed: {totals['negative_duration']:,}")
print(f"Rows retained: {totals['rows_retained']:,}")
print(f"Cleaned CSV files saved in: {output_dir}")

202608-citibike-tripdata_1.csv: 1,000,000 rows before; 791 over 3 hours removed; 0 negative-duration rides removed; 999,209 retained -> clean-202608-citibike-tripdata/202608-citibike-tripdata_1_cleaned.csv
202608-citibike-tripdata_2.csv: 1,000,000 rows before; 727 over 3 hours removed; 0 negative-duration rides removed; 999,273 retained -> clean-202608-citibike-tripdata/202608-citibike-tripdata_2_cleaned.csv
202608-citibike-tripdata_3.csv: 1,000,000 rows before; 1,904 over 3 hours removed; 0 negative-duration rides removed; 998,096 retained -> clean-202608-citibike-tripdata/202608-citibike-tripdata_3_cleaned.csv
202608-citibike-tripdata_4.csv: 1,000,000 rows before; 1,462 over 3 hours removed; 0 negative-duration rides removed; 998,538 retained -> clean-202608-citibike-tripdata/202608-citibike-tripdata_4_cleaned.csv
202608-citibike-tripdata_5.csv: 1,000,000 rows before; 961 over 3 hours removed; 0 negative-duration rides removed; 999,039 retained -> clean-202608-citibike-tripdata/20260